# QLoRA → evaluation → release gate → quantized vLLM serving (Colab, T4)

Runs the GPU pipeline of [arpitpaliwal007/qlora-vllm-finetuning](https://github.com/arpitpaliwal007/qlora-vllm-finetuning) on a Colab T4.

**Runtime → Change runtime type → T4 GPU** before running. Run the cells in order. Outputs (reports, manifests, adapter, quantized model) are synced to your Google Drive so a dead session does not lose them; cell 7 restores them to resume.

In [ ]:
#@title 1. GPU and driver check
!nvidia-smi
import subprocess, re
drv = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
cuda = re.search(r"CUDA Version: (\d+)\.(\d+)", drv)
print("Driver CUDA:", cuda.group(0) if cuda else "unknown")
assert cuda and int(cuda.group(1)) >= 12, "Need a CUDA 12+ driver (select a T4 GPU runtime)"


In [ ]:
#@title 2. Mount Google Drive and clone the repo
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/qlora-vllm-finetuning-runs'   # persistent outputs live here
WORK = '/content/qlora-vllm-finetuning'                       # fast local disk for the run
os.environ.update(DRIVE=DRIVE, WORK=WORK, TRAIN_PY=".venv-train/bin/python", SERVE_PY=".venv-serve/bin/python", PORT="8001")
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(WORK):
    !git clone https://github.com/arpitpaliwal007/qlora-vllm-finetuning {WORK}
%cd {WORK}
!git log --oneline -1


In [ ]:
%%bash
#@title 3. Two environments (train/quantize vs. serve) — ~10 min
# vLLM 0.32.0 pins compressed-tensors==0.17.0, llm-compressor 0.15.0 pins 0.20.1, so they cannot share one env.
# torch 2.13.0 cu129 wheels run on any CUDA-12 driver (minor-version compatibility).
cd "$WORK"
python -m venv .venv-train && .venv-train/bin/pip install -q --upgrade pip
.venv-train/bin/pip install -q --index-url https://download.pytorch.org/whl/cu129 torch==2.13.0
.venv-train/bin/pip install -q -r requirements-gpu-train.txt -e . pytest ruff
python -m venv .venv-serve && .venv-serve/bin/pip install -q --upgrade pip
.venv-serve/bin/pip install -q vllm==0.32.0 --extra-index-url https://download.pytorch.org/whl/cu129
.venv-serve/bin/pip install -q -e . --no-deps && .venv-serve/bin/pip install -q pyyaml numpy
.venv-train/bin/python -c "import torch;print('train env torch',torch.__version__,'cuda',torch.cuda.is_available(),torch.cuda.get_device_name(0))"
.venv-serve/bin/python -c "import vllm,torch;print('serve env vllm',vllm.__version__,'torch',torch.__version__)"


In [ ]:
%%bash
#@title 4. CPU test suite (must pass before any GPU work)
cd "$WORK"
.venv-train/bin/ruff check src scripts tests
.venv-train/bin/pytest -q


In [ ]:
%%bash
#@title 5. Smoke run — every stage, ~20 training steps (10–20 min)
cd "$WORK" && mkdir -p reports
set +e
bash scripts/run_gpu_pipeline.sh smoke 2>&1 | tee reports/smoke.log
rc=${PIPESTATUS[0]}
echo "pipeline exit code: $rc  (0 = release gate PASS; 1 = gate FAIL or a stage error - see reports/smoke/release_decision.json and the log)"


In [ ]:
%%bash
#@title 6. Sync outputs to Drive (run after every stage you care about)
# Keeps: reports, dataset manifests, the adapter (with checkpoints, for --resume) and the quantized model.
# Skips the merged fp16 model (~3 GB; re-creatable from the adapter in a minute) and vLLM logs.
cd "$WORK"
mkdir -p "$DRIVE/artifacts" "$DRIVE/data"
rsync -a reports/ "$DRIVE/reports/"
for d in data/*/; do mkdir -p "$DRIVE/$d" && cp "$d"manifest.json "$d"rejections.json "$DRIVE/$d" 2>/dev/null; done
for d in artifacts/qlora-adapter* artifacts/quantized-w4a16-*; do [ -d "$d" ] && rsync -a --exclude 'vllm.log' "$d/" "$DRIVE/$d/"; done
du -sh "$DRIVE"


In [ ]:
%%bash
#@title 7. (Only after a session restart) restore outputs from Drive to resume
cd "$WORK"
rsync -a "$DRIVE/reports/" reports/ 2>/dev/null; rsync -a "$DRIVE/artifacts/" artifacts/ 2>/dev/null; rsync -a "$DRIVE/data/" data/ 2>/dev/null || true
ls reports artifacts
# Resume an interrupted full training run, then re-run cell 8 (completed stages are skipped):
# .venv-train/bin/python scripts/train.py --config configs/qlora.yaml --resume


In [ ]:
%%bash
#@title 8. Full run — 20k examples × 2 epochs (several hours on a T4)
cd "$WORK" && mkdir -p reports
set +e
bash scripts/run_gpu_pipeline.sh full 2>&1 | tee reports/full.log
rc=${PIPESTATUS[0]}
echo "pipeline exit code: $rc  (0 = release gate PASS; 1 = gate FAIL or a stage error - see reports/full/release_decision.json and the log)"


In [ ]:
%%bash
#@title 9. Results, release decision, and the evidence files to commit
cd "$WORK"
cat reports/full/comparison/comparison.md 2>/dev/null || cat reports/smoke/comparison/comparison.md
.venv-train/bin/python - <<'EOF'
import json, glob
for p in sorted(glob.glob('reports/*/release_decision.json')):
    d = json.load(open(p)); print(p, '->', d['decision'], 'failed:', d['summary']['failed_checks'])
for p in sorted(glob.glob('reports/*/bench.json')):
    r = json.load(open(p))['results']
    print(p, 'p95 latency', r['request_latency_s'] and round(r['request_latency_s']['p95'], 3), 'tok/s', r['output_tokens_per_second'] and round(r['output_tokens_per_second'], 1))
EOF
# Evidence bundle for git (weights excluded): copy to Drive, then commit from your own machine.
E="$DRIVE/evidence"; mkdir -p "$E"
rsync -a --include '*/' --include 'comparison.*' --include 'release_decision.json' --include 'bench.json' --include 'results.json' --include '*.log' --exclude '*' reports/ "$E/reports/"
for d in data/*/; do mkdir -p "$E/$d" && cp "$d"manifest.json "$E/$d"; done
rsync -a --include '*/' --include 'training_metadata.json' --include 'artifact_manifest.json' --exclude '*' artifacts/ "$E/artifacts/"
cp configs/release_policy.yaml configs/qlora.yaml "$E/"
find "$E" -type f | sort


## After the run

1. The policy thresholds in `configs/release_policy.yaml` are illustrative. Calibrate them from `comparison.md` (base vs fine-tuned gap and bootstrap CIs), then re-run only the gate (README §6) if needed.
2. Commit the evidence files from `MyDrive/qlora-vllm-finetuning-runs/evidence` into the repo (never weights or `deploy/`), and fill the README "Status" table and a "Measured results" section with the numbers.
3. To serve the approved artifact elsewhere: `scripts/serve_vllm.sh start deploy/approved/current 8000`.